# 2ème partie du projet

In [1]:
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from mistralai.client import Mistral

from dotenv import load_dotenv
import os
load_dotenv(Path.cwd() / ".env")
api_key = os.environ["MISTRAL_API_KEY"]

In [5]:
# Création de la base Faiss et de l'index
# FAISS ne stocke que les vecteurs. On sauvegarde donc l’index FAISS
#  et un Parquet de métadonnées avec le même faiss_id.

# Colonnes à retourner avec les résultats de recherche.
colonnes_metadata_candidates = [
    "score_similarite",
    "uid",
    "title",
    "texte_chunk",
    "nextTiming",
    "lastTiming",
    "timings",
    "location.name",
    "location.city",
    "location.region",
    "location.address",
    "agenda_titre_source",
]

df_embeddings = pd.read_parquet("data/parquet_sortie/evenements_embeddings_mistral.parquet")
df_nettoye = pd.read_parquet("data/parquet_sortie/evenements_culturels_nettoye.parquet")

colonnes_metadata = [
    colonne
    for colonne in colonnes_metadata_candidates
    if colonne in df_nettoye.columns
]

# Traitement des métadonnées
# Associe chaque chunk à l'événement dont il provient.
metadata_evenements = df_nettoye[colonnes_metadata].copy()
metadata_evenements["index_evenement"] = metadata_evenements.index

metadata_faiss = (
    df_embeddings[["index_evenement", "numero_chunk", "texte_chunk"]]
    .merge(
        metadata_evenements,
        on="index_evenement",
        how="left",
        validate="many_to_one",
    )
    .reset_index(drop=True)
)

# Les IDs FAISS sont stables et correspondent aux lignes des métadonnées.
metadata_faiss.insert(
    0,
    "faiss_id",
    np.arange(len(metadata_faiss), dtype=np.int64),
)

# Matrice de vecteurs de type float32 et normalisation L2 = similarité cosinus.
vecteurs = np.asarray(
    df_embeddings["embedding"].tolist(),
    dtype=np.float32,
)

if vecteurs.ndim != 2 or len(vecteurs) != len(metadata_faiss):
    raise ValueError("Incohérence entre les embeddings et les métadonnées.")

faiss.normalize_L2(vecteurs)

dimension = vecteurs.shape[1]
index_faiss = faiss.IndexIDMap2(faiss.IndexFlatIP(dimension))

index_faiss.add_with_ids(
    vecteurs,
    metadata_faiss["faiss_id"].to_numpy(dtype=np.int64),
)

# Sauvegardes locales
dossier_index = Path("data/index_faiss")
dossier_index.mkdir(parents=True, exist_ok=True)

faiss.write_index(index_faiss, str(dossier_index / "evenements.faiss"))

metadata_faiss.drop(columns="embedding", errors="ignore").to_parquet(
    dossier_index / "metadata.parquet",
    index=False,
)

print(f"{index_faiss.ntotal} chunks indexés en dimension {dimension}.")

15647 chunks indexés en dimension 1024.


In [9]:
def rechercher_semantique(question, k=5):
    with Mistral(api_key=api_key) as mistral:
        reponse = mistral.embeddings.create(
            model="mistral-embed",
            inputs=[question],
        )

    vecteur_question = np.asarray(
        [reponse.data[0].embedding],
        dtype=np.float32,
    )
    faiss.normalize_L2(vecteur_question)

    scores, ids = index_faiss.search(vecteur_question, k)

    resultats = metadata_faiss.set_index("faiss_id").loc[ids[0]].copy()
    resultats["score_similarite"] = scores[0]

    return resultats[
        [
        "score_similarite",
        "uid",
        "title",
        "texte_chunk",
        "timings",
        "location.name",
        "location.city",
        "location.region",
        "location.address",
        "agenda_titre_source",
        ]
    ]

In [6]:
print(metadata_faiss.columns.tolist())

['faiss_id', 'index_evenement', 'numero_chunk', 'texte_chunk', 'uid', 'title', 'timings', 'location.name', 'location.city', 'location.region', 'location.address', 'agenda_titre_source']


In [14]:
resultats = rechercher_semantique(
    "Je cherche une pièce de théâtre à Rennes ce week-end",
    k=5,
)

display(resultats)

,score_similarite,uid,title,texte_chunk,timings,location.name,location.city,location.region,location.address,agenda_titre_source
faiss_id,,,,,,,,,,
11296,0.819763,90651645,Ateliers ludiques de jeux et exercices théâtra...,Aperçus disponibles en suivant les réseaux: \...,"[{""end"": ""2026-07-01T19:00:00+02:00"", ""begin"":...",Parc de Bréquigny,Rennes,Bretagne,Parc de Bréquigny,Rennes Métropole
10757,0.819763,90651645,Ateliers ludiques de jeux et exercices théâtra...,Aperçus disponibles en suivant les réseaux: \...,"[{""end"": ""2026-07-01T19:00:00+02:00"", ""begin"":...",Parc de Bréquigny,Rennes,Bretagne,Parc de Bréquigny,Direction éducation enfance
9100,0.819763,90651645,Ateliers ludiques de jeux et exercices théâtra...,Aperçus disponibles en suivant les réseaux: \...,"[{""end"": ""2026-07-01T19:00:00+02:00"", ""begin"":...",Parc de Bréquigny,Rennes,Bretagne,Parc de Bréquigny,MJC Bréquigny
12119,0.803000,12241501,"APRÈS NOUS, LES RUINES, PIERRE KOESTEL / LENA ...",Qui aurait pu sentir l’imminence d’une catastr...,"[{""end"": ""2026-11-25T20:15:00+01:00"", ""begin"":...",Théâtre National de Bretagne,Rennes,Bretagne,"1, rue saint-hélier",Rennes Métropole
830,0.803000,12241501,"APRÈS NOUS, LES RUINES, PIERRE KOESTEL / LENA ...",Qui aurait pu sentir l’imminence d’une catastr...,"[{""end"": ""2026-11-25T20:15:00+01:00"", ""begin"":...",Théâtre National de Bretagne,Rennes,Bretagne,"1, rue saint-hélier",Théâtre National de Bretagne
